In [3]:
# Imports

import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# load_dotenv()

# TOKEN = os.getenv("AQICN_TOKEN")
CITY = "Lahore"
DAYS_TO_FETCH    = 90


In [5]:
# fetching air quality data from open meteo
def get_city_coordinates(city_name):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
    response = requests.get(url)
    data = response.json()
    if "results" in data:
        return data["results"][0]["latitude"], data["results"][0]["longitude"], data["results"][0]["name"]
    else:
        raise ValueError(f"City '{city_name}' not found.")

lat, lon, city = get_city_coordinates(CITY)
print(f"Coordinates for {city}: Lat={lat}, Lon={lon}")

def fetch_aqi_data_openmeteo(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # meteo's api endpoint
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust",
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "timezone": "auto"
    }
    
    print(f"Air Quality data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()
    
    if "hourly" not in data:
        raise ValueError(f"Open-Meteo AQI API Error: {data}")
    
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "us_aqi": data["hourly"]["us_aqi"],
        "pm25": data["hourly"]["pm2_5"],
        "pm10": data["hourly"]["pm10"],
        "co": data["hourly"]["carbon_monoxide"],
        "no2": data["hourly"]["nitrogen_dioxide"],
        "so2": data["hourly"]["sulphur_dioxide"],
        "o3": data["hourly"]["ozone"],
        "dust": data["hourly"]["dust"]
    })
    
    return df

df_aqi = fetch_aqi_data_openmeteo(lat, lon, DAYS_TO_FETCH)
print(f"Fetched {len(df_aqi)} records.")
display(df_aqi.head(10))

Coordinates for Lahore: Lat=31.558, Lon=74.35071
Air Quality data from 2026-04-27 to 2026-07-26...
Fetched 2184 records.


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust
0,2026-04-27 00:00:00,77,20.4,27.1,467.0,30.7,6.6,61.0,11.0
1,2026-04-27 01:00:00,76,22.8,30.3,422.0,31.7,6.8,56.0,12.0
2,2026-04-27 02:00:00,74,24.0,31.5,395.0,32.7,7.0,51.0,13.0
3,2026-04-27 03:00:00,73,25.1,32.1,387.0,34.1,7.2,41.0,12.0
4,2026-04-27 04:00:00,72,26.4,32.5,396.0,35.6,7.4,32.0,10.0
5,2026-04-27 05:00:00,71,29.7,38.9,446.0,35.1,7.4,43.0,15.0
6,2026-04-27 06:00:00,70,34.0,43.7,588.0,35.6,8.0,58.0,14.0
7,2026-04-27 07:00:00,69,38.9,45.5,769.0,36.0,9.0,80.0,13.0
8,2026-04-27 08:00:00,69,27.0,35.0,858.0,33.3,9.8,104.0,13.0
9,2026-04-27 09:00:00,68,18.3,27.1,768.0,24.8,10.6,131.0,15.0


In [6]:
# getting weather data for accurate prediction of next days as current air quality not enough to predict future.
def fetch_weather_data(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }
    
    print(f"Weather data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()
    
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "temperature_c": data["hourly"]["temperature_2m"],
        "humidity_pct": data["hourly"]["relative_humidity_2m"],
        "wind_speed_kmh": data["hourly"]["wind_speed_10m"],
        "precipitation_mm": data["hourly"]["precipitation"]
    })
    
    return df

df_weather = fetch_weather_data(lat, lon, DAYS_TO_FETCH)

# merge aqi and weather df for 1 single df
df_raw = pd.merge(df_aqi, df_weather, on="timestamp", how="inner")
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)

print(f"Final raw dataset shape: {df_raw.shape}")
display(df_raw.head(10))

Weather data from 2026-04-27 to 2026-07-26...
Final raw dataset shape: (2184, 13)


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust,temperature_c,humidity_pct,wind_speed_kmh,precipitation_mm
0,2026-04-27 00:00:00,77,20.4,27.1,467.0,30.7,6.6,61.0,11.0,30.3,31,7.8,0.0
1,2026-04-27 01:00:00,76,22.8,30.3,422.0,31.7,6.8,56.0,12.0,29.2,34,7.3,0.0
2,2026-04-27 02:00:00,74,24.0,31.5,395.0,32.7,7.0,51.0,13.0,28.0,37,6.0,0.0
3,2026-04-27 03:00:00,73,25.1,32.1,387.0,34.1,7.2,41.0,12.0,27.3,38,4.4,0.0
4,2026-04-27 04:00:00,72,26.4,32.5,396.0,35.6,7.4,32.0,10.0,27.1,37,5.1,0.0
5,2026-04-27 05:00:00,71,29.7,38.9,446.0,35.1,7.4,43.0,15.0,27.2,38,1.8,0.0
6,2026-04-27 06:00:00,70,34.0,43.7,588.0,35.6,8.0,58.0,14.0,26.8,38,3.2,0.0
7,2026-04-27 07:00:00,69,38.9,45.5,769.0,36.0,9.0,80.0,13.0,28.1,35,5.9,0.0
8,2026-04-27 08:00:00,69,27.0,35.0,858.0,33.3,9.8,104.0,13.0,30.6,31,3.1,0.0
9,2026-04-27 09:00:00,68,18.3,27.1,768.0,24.8,10.6,131.0,15.0,33.6,25,2.2,0.0
